In [ ]:
!pip install -q "transformers==4.57.3" tokenizers accelerate python-multipart


In [ ]:
!!pip install "surya-ocr==0.17.1"

['Collecting surya-ocr==0.17.1',
 '  Downloading surya_ocr-0.17.1-py3-none-any.whl.metadata (34 kB)',
 'Requirement already satisfied: click<9.0.0,>=8.1.8 in /usr/local/lib/python3.12/dist-packages (from surya-ocr==0.17.1) (8.4.1)',
 'Requirement already satisfied: einops<0.9.0,>=0.8.1 in /usr/local/lib/python3.12/dist-packages (from surya-ocr==0.17.1) (0.8.2)',
 'Requirement already satisfied: filetype<2.0.0,>=1.2.0 in /usr/local/lib/python3.12/dist-packages (from surya-ocr==0.17.1) (1.2.0)',
 'Requirement already satisfied: opencv-python-headless==4.11.0.86 in /usr/local/lib/python3.12/dist-packages (from surya-ocr==0.17.1) (4.11.0.86)',
 'Requirement already satisfied: pillow<11.0.0,>=10.2.0 in /usr/local/lib/python3.12/dist-packages (from surya-ocr==0.17.1) (10.4.0)',
 'Requirement already satisfied: platformdirs<5.0.0,>=4.3.6 in /usr/local/lib/python3.12/dist-packages (from surya-ocr==0.17.1) (4.10.0)',
 'Collecting pre-commit<5.0.0,>=4.2.0 (from surya-ocr==0.17.1)',
 '  Downloadi

In [ ]:
!pip install -q sentence-transformers

In [ ]:
!pip install pdf2image

In [ ]:
!pip install -q pyngrok

In [ ]:
# Cell 5 — Load Surya v1 models ONCE (in-process, GPU, no vLLM)
from surya.foundation import FoundationPredictor
from surya.recognition import RecognitionPredictor
from surya.detection import DetectionPredictor

foundation = FoundationPredictor()
detector   = DetectionPredictor()
recognizer = RecognitionPredictor(foundation)

# Optional embedding model. The SME-GPT backend embeds locally, so /embed below
# is unused by the live pipeline — safe to delete this + the /embed route to save VRAM.
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("intfloat/multilingual-e5-small")

print("Models loaded OK")


In [ ]:
# Cell 6 — OCR server matching the SME-GPT backend contract
#
# POST /ocr  (multipart form-data, multiple parts per field, one per page):
#     orig   - deskewed RGB image       (required)
#     p_img  - binarized variant        (optional)
#     m_img  - bilateral-filtered       (optional)
#   Returns: { success, engine, versions: {orig,P,M: {text, pages:[{page,text,text_lines}]}}, failures }
#   Each text_line: { text, confidence, polygon, bbox }
#
# GET /health -> { status, engine, mode }

import io
from typing import List, Dict, Any

from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image

NL = chr(10)

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"], allow_credentials=True,
    allow_methods=["*"], allow_headers=["*"],
)

SURYA_MODE = "surya-0.17.1 (v1 in-process/cuda)"


def _ocr_images(images: List[Image.Image]) -> List[Dict[str, Any]]:
    """Run Surya v1 detection+recognition on a batch of page images.
    Returns [{text, text_lines:[{text,confidence,polygon,bbox}]}] (one per page)."""
    if not images:
        return []
    rgb = [im.convert("RGB") if im.mode != "RGB" else im for im in images]
    results = recognizer(rgb, det_predictor=detector)   # batched across pages

    pages = []
    for page_res in results:
        if hasattr(page_res, "model_dump"):
            pd = page_res.model_dump()
        elif hasattr(page_res, "dict"):
            pd = page_res.dict()
        else:
            pd = dict(page_res)

        lines = []
        for ln in pd.get("text_lines", []) or []:
            txt = (ln.get("text") or "").strip()
            if not txt:
                continue
            lines.append({
                "text"      : txt,
                "confidence": float(ln.get("confidence") or 0.0),
                "polygon"   : ln.get("polygon"),
                "bbox"      : ln.get("bbox"),
            })
        page_text = NL.join(l["text"] for l in lines).strip()
        pages.append({"text": page_text, "text_lines": lines})
    return pages


def _build_version(images: List[Image.Image], vname: str, failures: dict) -> dict:
    """OCR all pages of one variant -> {text, pages:[{page,text,text_lines}]}."""
    if not images:
        return {"text": "", "pages": []}
    try:
        page_results = _ocr_images(images)
    except Exception as exc:
        failures[vname] = str(exc)
        print("  [OCR] variant " + vname + " failed: " + str(exc), flush=True)
        return {"text": "", "pages": []}

    n = len(page_results)
    pages_out, parts = [], []
    for i, pr in enumerate(page_results, start=1):
        pages_out.append({"page": i, "text": pr["text"], "text_lines": pr["text_lines"]})
        if pr["text"]:
            parts.append(("=== PAGE " + str(i) + " ===" + NL + pr["text"]) if n > 1 else pr["text"])
    return {"text": (NL + NL).join(parts).strip(), "pages": pages_out}


async def _read_images(files: List[UploadFile]) -> List[Image.Image]:
    imgs = []
    for f in files:
        raw = await f.read()
        if raw:
            imgs.append(Image.open(io.BytesIO(raw)))
    return imgs


@app.get("/health")
def health():
    return {"status": "ok", "engine": "surya", "mode": SURYA_MODE}


@app.post("/ocr")
async def ocr(
    orig:  List[UploadFile] = File(default=[]),
    p_img: List[UploadFile] = File(default=[]),
    m_img: List[UploadFile] = File(default=[]),
):
    failures: Dict[str, Any] = {}

    orig_imgs = await _read_images(orig)
    if not orig_imgs:
        return {"success": False, "error": "No 'orig' images received.",
                "engine": "colab_surya_v1", "versions": {}, "failures": failures}

    p_imgs = await _read_images(p_img)
    m_imgs = await _read_images(m_img)
    print("[OCR] pages -> orig:" + str(len(orig_imgs)) + " P:" + str(len(p_imgs)) + " M:" + str(len(m_imgs)), flush=True)

    versions = {
        "orig": _build_version(orig_imgs, "orig", failures),
        "P":    _build_version(p_imgs,    "P",    failures),
        "M":    _build_version(m_imgs,    "M",    failures),
    }
    return {"success": True, "engine": "colab_surya_v1",
            "versions": versions, "failures": failures}


@app.post("/embed")
async def embed(payload: dict):
    texts = payload.get("texts", [])
    mode = payload.get("mode", "passage")
    if mode not in ("passage", "query"):
        mode = "passage"
    prefixed = [(mode + ": " + t) for t in texts]
    vecs = embedder.encode(prefixed, normalize_embeddings=True)
    return {"embeddings": vecs.tolist(), "dim": len(vecs[0]) if len(vecs) else 0}


# Run server


In [ ]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn
from google.colab import userdata

# 1. Apply fix for running nested event loops (required for Colab)
nest_asyncio.apply()

# 2. Setup ngrok
# It's safer to use Colab Secrets for your token
# In the sidebar, click the Key icon, add 'NGROK_AUTH_TOKEN'
try:
    NGROK_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
    ngrok.set_auth_token(NGROK_TOKEN)
except:
    print("Warning: NGROK_AUTH_TOKEN not found in Secrets. Using default (if set).")

# 3. Create Tunnel
# bind_tls=True ensures you get an https:// link
public_url = ngrok.connect(8000, bind_tls=True).public_url
print("\n" + "="*50)
print(f"🚀 OCR Worker is LIVE at: {public_url}")
print("="*50 + "\n")

# 4. Start the Server
# This will block the cell, but the tunnel is already up!
import asyncio

# Create a config object instead of calling run directly
config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)

# Run the server on the existing Colab loop
await server.serve()


🚀 OCR Worker is LIVE at: https://nonentomologic-suffruticose-donya.ngrok-free.dev



INFO:     Started server process [2688]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
